<!--
Licensed to the Apache Software Foundation (ASF) under one or more
contributor license agreements.  See the NOTICE file distributed with
this work for additional information regarding copyright ownership.
The ASF licenses this file to You under the Apache License, Version 2.0
(the "License"); you may not use this file except in compliance with
the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 03 · Experiment 3 — Basic + Wren (base layer)

Resolve the Seagate semantic project and **onboard** it. Onboarding seeds the model structure from catalog introspection and **auto-activates** the base models (no business semantics yet). The agent now has structural grounding — table/column awareness and join structure — but not the glossary's meanings.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import eval_common as ec

# Edit here if your endpoints differ (Docker: agent_base_url=
# 'http://localhost:8090/ai-agent', superset_base_url='http://localhost:8090').
config = ec.EvalConfig.from_env()
client = ec.AgentClient(config)
me = client.login()
print('Authenticated as:', me.get('username') or me.get('email') or me)
print('DB id:', client.resolve_database_id(), '| schema:', config.schema_name)
questions = ec.parse_test_queries()
print('Loaded', len(questions), 'graded questions')

### Resolve + onboard (auto-activates base models)

In [ ]:
project = client.resolve_project()
print('Project:', project['id'])
job = client.onboard(project['id'])
result = job.get('result') or {}
print('Onboarding:', result.get('model_count'), 'models,',
      result.get('activated_count'), 'activated')
print('Active MDL files:', sum(1 for f in client.list_mdl_files(project['id'])
      if f.get('status') == 'active'))

### Run the 15 questions against the base Wren layer

In [ ]:
results = ec.run_experiment(client, 'wren_base', questions)
for r in results:
    print(f"{r['id']:>3} [{r['status']}] models={r['wren'].get('matched_models')} "
          f"{str(r['answer_summary'])[:70]}")

### Grade vs. ground truth

In [ ]:
graded = ec.grade_experiment(questions, results)
print('Score summary:', ec.score_summary(graded))
for g in graded:
    print(f"{g['id']:>3} {g['level']} {g['verdict']:<14} expected={g['expected']!r} got={str(g['got'])[:80]!r}")